### Preparation Data

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten,Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

Adam = tf.keras.optimizers.Adam
to_categorical = tf.keras.utils.to_categorical
array_to_img = tf.keras.preprocessing.image.array_to_img
ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau

import numpy as np
import pandas as pd
import os
import cv2
from collections import Counter

from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt

import plotly
import plotly.graph_objs as go
import plotly.figure_factory as ff
import plotly.express as px




In [2]:
img_rows , img_cols = 224 , 224
batch_size = 16
epochs = 100
num_classes = 2
seed = 1

input_shape = (img_rows , img_cols , 3)
devide_list = tf.test.gpu_device_name()


In [3]:
train_path = "D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Dior/train"
test_path = "D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Dior/test"
val_path = "D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Dior/val"

In [4]:
x_train = []

for folder in os.listdir(train_path) :

    sub_path = train_path + "/" + folder

    for img in os.listdir(sub_path) :

        image_path = sub_path + "/" + img

        img_arr = cv2.imread(image_path)

        img_arr = cv2.resize(img_arr , (224 , 224))

        x_train.append(img_arr)


In [5]:
x_test = []

for folder in os.listdir(test_path) :

    sub_path = test_path + "/" + folder

    for img in os.listdir(sub_path) :

        image_path = sub_path + "/" + img

        img_arr = cv2.imread(image_path)

        img_arr = cv2.resize(img_arr , (224 , 224))

        x_test.append(img_arr)

In [6]:
x_val = []

for folder in os.listdir(val_path) :

    sub_path = val_path + "/" + folder

    for img in os.listdir(sub_path) :

        image_path = sub_path + "/" + img

        img_arr = cv2.imread(image_path)

        img_arr = cv2.resize(img_arr , (224 , 224))

        x_val.append(img_arr)

In [7]:
train_x = np.array(x_train)
test_x = np.array(x_test)
val_x = np.array(x_val)

In [8]:
train_x = train_x / 255.0
test_x = test_x / 255.0
val_x = val_x / 255.0

### Augmentation

In [29]:
train_datagen = ImageDataGenerator(rotation_range = 0.05 ,
                                   width_shift_range = 0.2 ,
                                   height_shift_range = 0.2 ,
                                   zoom_range = 0.3 ,
                                   horizontal_flip = True)

test_datagen = ImageDataGenerator(rescale = 1. / 255)

val_datagen = ImageDataGenerator(rescale = 1. / 255)

In [30]:
train_set = train_datagen.flow_from_directory(train_path , 
                                            target_size = (img_rows , img_cols) ,
                                            batch_size = batch_size ,
                                            class_mode = 'binary' ,
                                            shuffle = True ,
                                            color_mode = 'rgb' ,
                                            seed = seed)

test_set = test_datagen.flow_from_directory(test_path ,
                                            target_size = (img_rows , img_cols) ,
                                            batch_size = batch_size ,
                                            class_mode = 'binary' ,
                                            shuffle = True ,
                                            color_mode = 'rgb' ,
                                            seed = seed)

val_set = val_datagen.flow_from_directory(val_path , 
                                            target_size = (img_rows , img_cols) ,
                                            batch_size = batch_size ,
                                            class_mode = 'binary' ,
                                            shuffle = True ,
                                            color_mode = 'rgb' ,
                                            seed = seed)

Found 316 images belonging to 2 classes.
Found 47 images belonging to 2 classes.
Found 89 images belonging to 2 classes.


In [31]:
batchX, batchY = train_set.next()
print('Batch shape = %s , min = %.3f , max = %.3f' % (batchX.shape , batchX.min() , batchX.max()))

Batch shape = (16, 224, 224, 3) , min = 0.000 , max = 255.000


In [32]:
labels = list(train_set.class_indices.keys())
print(labels)

['fake', 'real']


In [33]:
print(Counter(train_set.classes))

Counter({1: 160, 0: 156})


In [34]:
weights = compute_class_weight('balanced' , classes =  np.unique(train_set.classes) , y = train_set.classes)
print(weights)

[1.01282051 0.9875    ]


In [35]:
cw = dict(zip(np.unique(train_set.classes) , weights))
print(cw)

{0: 1.0128205128205128, 1: 0.9875}


In [36]:
# model = Sequential()
model = tf.keras.Sequential()

#1. CNN LAYER
model.add(tf.keras.layers.Conv2D(filters = 32 , kernel_size = (3 , 3) , padding = 'Same' , input_shape = input_shape))
model.add(tf.keras.layers.Activation("relu"))

#2. CNN LAYER
model.add(tf.keras.layers.Conv2D(filters = 32 , kernel_size = (3 , 3) , padding = 'Same'))
model.add(tf.keras.layers.Activation("relu"))

model.add(tf.keras.layers.MaxPool2D(pool_size = (2 , 2)))

#3. CNN LAYER
model.add(tf.keras.layers.Conv2D(filters = 64 , kernel_size = (3 , 3) , padding = 'Same'))
model.add(tf.keras.layers.Activation("relu"))

#4. CNN LAYER
model.add(tf.keras.layers.Conv2D(filters = 64 , kernel_size = (3 , 3) , padding = 'Same'))
model.add(tf.keras.layers.Activation("relu"))

model.add(tf.keras.layers.MaxPool2D(pool_size = (2 , 2)))

#Image Classification
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(128 , activation = 'relu'))
model.add(tf.keras.layers.Dense(64 , activation = 'relu'))
model.add(tf.keras.layers.Dense(1 , activation = 'sigmoid'))


In [37]:
STEP_SIZE_TRAIN = train_set.n // train_set.batch_size
STEP_SIZE_VALID = val_set.n // val_set.batch_size
STEP_SIZE_TEST = test_set.n // test_set.batch_size

In [38]:
model.compile(loss = 'binary_crossentropy' , optimizer = 'adam' , metrics = ['accuracy'])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 224, 224, 32)      896       
                                                                 
 activation_4 (Activation)   (None, 224, 224, 32)      0         
                                                                 
 conv2d_5 (Conv2D)           (None, 224, 224, 32)      9248      
                                                                 
 activation_5 (Activation)   (None, 224, 224, 32)      0         
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 112, 112, 32)     0         
 2D)                                                             
                                                                 
 conv2d_6 (Conv2D)           (None, 112, 112, 64)      18496     
                                                      

In [39]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_loss' , patience = 2 , verbose = 1 ,factor = 0.1 , min_lr = 0.000001)
callbacks_list = [learning_rate_reduction]

In [40]:
his = model.fit(
        train_set ,
        validation_data = val_set ,
        epochs = epochs ,
        class_weight = cw ,
        callbacks = callbacks_list ,
        shuffle = True ,
        steps_per_epoch = STEP_SIZE_TRAIN ,
        validation_steps = STEP_SIZE_VALID)

Epoch 1/100
19/19 [==============================] - 31s 2s/step - loss: 62.9188 - accuracy: 0.4733 - val_loss: 0.6931 - val_accuracy: 0.5125 - lr: 0.0010
Epoch 2/100
19/19 [==============================] - 28s 1s/step - loss: 0.6887 - accuracy: 0.5433 - val_loss: 0.6934 - val_accuracy: 0.4750 - lr: 0.0010
Epoch 3/100
19/19 [==============================] - 28s 1s/step - loss: 0.6891 - accuracy: 0.5467 - val_loss: 0.6930 - val_accuracy: 0.5125 - lr: 0.0010
Epoch 4/100
19/19 [==============================] - 29s 1s/step - loss: 0.6812 - accuracy: 0.5700 - val_loss: 0.6933 - val_accuracy: 0.4750 - lr: 0.0010
Epoch 5/100
19/19 [==============================] - ETA: 0s - loss: 0.6971 - accuracy: 0.5200
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
19/19 [==============================] - 29s 2s/step - loss: 0.6971 - accuracy: 0.5200 - val_loss: 0.6931 - val_accuracy: 0.5125 - lr: 0.0010
Epoch 6/100
19/19 [==============================] - 29s 2s/step - lo

: 

: 

In [43]:
h1 = go.Scatter(y = his.history['loss'] , 
                    mode = "lines" ,
                    line = dict(
                        width = 2 ,
                        color = 'blue') ,
                        name = "loss"
                   )

h2 = go.Scatter(y = his.history['val_loss'] , 
                    mode = "lines" ,
                    line = dict(
                        width = 2 ,
                        color = 'red') ,
                        name = "val_loss"
                   )
                   
data = [h1 , h2]
layout1 = go.Layout(title = 'Loss' ,
                   xaxis = dict(title = 'epochs') ,
                   yaxis = dict(title = ''))
fig1 = go.Figure(data , layout = layout1)
plotly.offline.iplot(fig1)

In [44]:
h1 = go.Scatter(y = his.history['accuracy'] , 
                    mode = "lines" , 
                    line=dict(
                        width = 2 ,
                        color = 'blue') ,
                        name = "acc"
                   )

h2 = go.Scatter(y = his.history['val_accuracy'] , 
                    mode = "lines" , 
                    line = dict(
                        width = 2 ,
                        color = 'red') ,
                        name = "val_acc"
                   )

data = [h1 , h2]
layout1 = go.Layout(title = 'Accuracy' ,
                   xaxis = dict(title = 'epochs') ,
                   yaxis = dict(title = ''))
fig1 = go.Figure(data = data , layout = layout1)
plotly.offline.iplot(fig1)

In [45]:
score = model.evaluate(test_set , verbose = 0)
print("Test Loss : " , score[0])
print("Test Accuracy : " , score[1])

Test Loss :  0.6931897401809692
Test Accuracy :  0.4680851101875305


### Evaluation

In [ ]:
predicted_classes = (model.predict(test_set , verbose = 1) > 0.5).astype("int32")[ : , 0]

In [ ]:
def cm_plot(cm , labels) :
    x = labels
    y = labels

    z_text = [[str(y) for y in x] for x in cm]
    fig = ff.create_annotated_heatmap(cm , x = x , y = y , annotation_text = z_text , colorscale = 'blues')

    fig.update_layout(title_text = 'Confusion Matrix')

    fig.add_annotation(dict(font = dict(color = "black" , size = 20) ,
                            x = 0.5 ,
                            y = -0.15 ,
                            showarrow = False ,
                            text = "Predicted Value" ,
                            xref = "paper" ,
                            yref = "paper"
                           ))

    fig.add_annotation(dict(font=dict(color = "black" , size = 20),
                            x = -0.20 ,
                            y = 0.5 ,
                            showarrow = False ,
                            text = "Real Value" ,
                            textangle = -90 ,
                            xref = "paper" ,
                            yref = "paper"
                           ))


    fig.update_layout(margin = dict(t = 50 , l = 200))
    fig['layout']['yaxis']['autorange'] = "reversed"

    fig['data'][0]['showscale'] = True
    fig.show()

In [ ]:
cm = confusion_matrix(test_set.classes , predicted_classes)
cm

In [ ]:
cm_plot(cm , labels)

In [ ]:
report = classification_report(test_set.classes , predicted_classes , target_names = labels , digits = 4)

print(report)